# VadCLIP + Khuếch Đại Mất Mát Theo Lớp — Vòng 2

Notebook này là bản làm lại sau khi mất dữ liệu Drive. Nó **không** phải bản sao của vòng 1:
thiết kế đã được sửa dựa trên những gì vòng 1 đo được. Xem `docs/HANDOFF_RescaleEWC.md` để
biết chi tiết số liệu vòng 1.

## Sáu thay đổi so với vòng 1, và lý do

| Thay đổi | Lý do rút ra từ vòng 1 |
|---|---|
| **Bỏ nhánh Fisher/EWC khỏi đường chạy chính** | Ba cơ chế bảo vệ cho 62,41 / 62,63 / 62,79 — không phân biệt được. Độ trôi trọng số chỉ ~1e-5 nên không có gì để bảo vệ |
| **Không chọn checkpoint theo tập test** (`--select-metric none`) | Vòng 1 lấy cực đại của ~39 lần chấm trên tập test → ước lượng thiên lệch lên. Giờ lấy thẳng trọng số cuối |
| **Chỉ đánh giá cuối mỗi epoch** (`--eval-steps 0`) | Bỏ chọn checkpoint rồi thì không cần chấm 39 lần. Còn 3 lần/run — tiết kiệm ~90% thời gian đánh giá |
| **Dải μ đổi thành 3, 5, 8, 12** | μ=5 nằm **dưới ngưỡng phân giải**: +1,11 ở seed này nhưng +0,09 ở seed kia. μ=8 mới lặp lại được |
| **Hai seed ngay từ đầu, mỗi seed có ctrl riêng** | Bài học lớn nhất vòng 1. Ba lần chạy dùng chung seed cho ước lượng nhiễu 0,38 — sai, nhiễu thật giữa hai seed là ~1,2 |
| **Thêm mục 4.1 sinh lại ground truth** | Ba file `gt_*.npy` chỉ từng có trên Drive và đã mất. Hai script gốc `make_gt_ucf.py` / `make_gt_mAP_ucf.py` không chạy được: chúng lọc theo `__0.npy` trong khi list dùng `__5.npy` |
| **Chế độ video chỉ chạy 1 lần thay vì 3** | Đã kết luận chắc: chế độ lớp thắng chế độ video ở cả hai chiều (AP nhóm đích của video đi xuống, tới −0,49 ở μ=5) |

Tổng: **9 lần chạy** thay vì 15, mỗi lần nhanh hơn đáng kể.

## Điều đã biết chắc từ vòng 1

- Mức tăng thêm khi đi từ μ=5 lên μ=8 là **+0,85 (seed 234)** và **+0,82 (seed 1234)** trên
  AUC nhóm đích. Lặp lại gần như trùng khít — đây là bằng chứng vững nhất.
- Ước lượng trung bình hai seed: μ=5 cho **+0,60**, μ=8 cho **+1,44**.
- Cái giá: Shooting bị hại nhất quán (7/7 lần chạy ở μ≥5 đều âm), báo động giả tăng 0,5–0,9.
- Đường đáp ứng **chưa gãy** ở μ=8 → mục đích chính của vòng này là tìm điểm gãy.

## 1. Mount Drive Và Cấu Hình

**Lưu ý sau sự cố mất dữ liệu:** phải upload lại toàn bộ thư mục `VadCLIP/src_rescale_ewc/`
từ repo cục bộ lên Drive. Bản cục bộ có các tham số mới `--run-tag` và `--select-metric` mà
bản cũ trên Drive không có; thiếu chúng thì notebook này lỗi ngay ở lần chạy đầu.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src_rescale_ewc'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT

# theta': checkpoint giai doan 1 da hoi tu. BAT BUOC phai co.
STAGE1_MODEL = PROJECT_ROOT / 'model_ucf.pth'

RESULT_DIR = PROJECT_ROOT / 'Result'
LOG_DIR = RESULT_DIR / 'logs_rescale_ewc_v2'
MODEL_DIR = Path('model')

TRAIN_LIST = '../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST = '../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]

FISHER_PATH = 'model/fisher_ucf.pt'
METRICS_CSV = str(RESULT_DIR / 'rescale_ewc_v2_metrics.csv')
PERCLASS_CSV = RESULT_DIR / 'rescale_ewc_v2_perclass.csv'

# Giu nguyen nhom dich cua vong 1 de so sanh duoc voi so lieu da ghi trong bao cao.
# Thu nghiem thu hep con {Explosion, Shoplifting} da THAT BAI o vong 1 (Explosion chi +0,41
# so voi +3,16 khi giu ca bon lop), nen khong lap lai.
TARGET_CLASSES = ['Explosion', 'RoadAccidents', 'Shooting', 'Shoplifting']

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_stage2_cmd(tag, rescale_mode='class', mu=1.0, regularizer='none',
                     lambda_auto=0.0, lambda_reg=0.0, max_epoch=3, lr='2e-6',
                     oversample=1.0, seed=234, select_metric='none', eval_steps=0,
                     extra=None):
    """Dung lenh train giai doan 2.

    Ba mac dinh doi so voi vong 1, deu co ly do do duoc:
      regularizer='none'    Fisher khong dong gop gi (62,41 / 62,63 / 62,79)
      select_metric='none'  giu trong so CUOI, khong chon dinh tren tap test
      eval_steps=0          chi cham diem cuoi moi epoch, 3 lan thay vi 39
    """
    return PY + [
        'ucf_train_rescale.py',
        '--pretrained-model-path', STAGE1_MODEL,
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--fisher-path', FISHER_PATH,
        '--target-classes', *TARGET_CLASSES,
        '--seed', seed,
        '--rescale-mode', rescale_mode,
        '--mu', mu,
        '--regularizer', regularizer,
        '--lambda-reg', lambda_reg,
        '--lambda-auto', lambda_auto,
        '--target-oversample', oversample,
        '--max-epoch', max_epoch,
        '--lr', lr,
        '--batch-size', 64,
        '--num-workers', 4,
        '--pin-memory', 'true',
        '--eval-steps', eval_steps,
        '--select-metric', select_metric,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV,
        '--output-model-path', f'model/v2_{tag}.pth',
        '--checkpoint-path', f'model/checkpoint_v2_{tag}.pth',
        '--save-cur-path', f'model/model_cur_v2_{tag}.pth',
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_v2_{tag}',
    ] + list(extra or [])


def train_stage2(tag, **kwargs):
    # Tien to v2_ co chu dich: checkpoint vong 1 dung tien to s2_. Neu chung con tren
    # Drive thi co che bo qua duoi day se dung lai chung — nhung chung duoc huan luyen
    # voi thiet lap CU (co Fisher, co chon dinh tren tap test), tron vao bang ket qua
    # moi thi khong the phat hien tu so lieu.
    if Path(f'model/v2_{tag}.pth').exists():
        print(f'[bo qua] model/v2_{tag}.pth da ton tai. Xoa file neu muon chay lai.')
        return None
    return run_command(build_stage2_cmd(tag, **kwargs), log_name=f'train_v2_{tag}.log')


print('Project root :', PROJECT_ROOT)
print('Source dir   :', SRC_DIR)
print('Stage-1 model:', STAGE1_MODEL, '| exists:', STAGE1_MODEL.exists())
print('Target set   :', TARGET_CLASSES)

## 2. Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas

## 3. Kiểm Tra File Bắt Buộc

Cell này kiểm tra hai điều. Thứ nhất là các file dữ liệu và mô hình có mặt đủ chưa. Thứ hai,
và quan trọng sau sự cố mất dữ liệu: **bản code trên Drive có phải bản mới không**. Nếu
`--select-metric` không tồn tại thì bạn đang dùng bản cũ và phải upload lại
`VadCLIP/src_rescale_ewc/` từ repo cục bộ.

Ba file ground truth `gt_*.npy` được kiểm tra **riêng** và không làm dừng cell, vì nếu
thiếu thì mục 4.1 sinh lại được — nhưng việc đó cần feature nên phải làm sau mục 4.

Đừng dùng `list/make_gt_ucf.py` và `list/make_gt_mAP_ucf.py` gốc: chúng lọc theo `__0.npy`
trong khi mọi file list test của dự án dùng `__5.npy`, nên chạy ra file rỗng; và chúng đọc
CSV chứa đường dẫn tuyệt đối trên máy tác giả gốc.

In [ ]:
required_paths = [
    SRC_DIR / 'losses.py', SRC_DIR / 'fisher.py', SRC_DIR / 'evaluation.py',
    SRC_DIR / 'dataset_rescale.py', SRC_DIR / 'ucf_option_rescale.py',
    SRC_DIR / 'ucf_fisher.py', SRC_DIR / 'ucf_train_rescale.py',
    SRC_DIR / 'ucf_eval_perclass.py', SRC_DIR / 'tests' / 'test_losses.py',
    PROJECT_ROOT / 'VadCLIP' / 'src' / 'model.py',
    PROJECT_ROOT / 'VadCLIP' / 'src' / 'utils' / 'tools.py',
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    LIST_DIR / 'Temporal_Anomaly_Annotation.txt',
    LIST_DIR / 'make_gt_ucf_relative.py',
    STAGE1_MODEL,
]
missing = [p for p in required_paths if not p.exists()]
for p in required_paths:
    print(('OK   ' if p.exists() else 'MISS '), p)
if missing:
    raise FileNotFoundError(f'{len(missing)} file bat buoc chua co tren Drive.')

# Ground truth kiem tra rieng: thieu thi muc 4.1 sinh lai duoc.
GT_FILES = [LIST_DIR / n for n in ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')]
GT_MISSING = [p for p in GT_FILES if not p.exists()]
print()
for p in GT_FILES:
    print(('OK   ' if p.exists() else 'THIEU'), p)
if GT_MISSING:
    print()
    print(f'-> Thieu {len(GT_MISSING)} file ground truth. Chay muc 4 roi muc 4.1 de sinh lai.')

# Kiem tra ban code moi.
opts = (SRC_DIR / 'ucf_option_rescale.py').read_text(encoding='utf-8')
for flag in ('--select-metric', '--run-tag'):
    if flag not in opts:
        raise RuntimeError(
            f'{flag} khong co trong ucf_option_rescale.py. Ban code tren Drive la ban CU. '
            'Upload lai toan bo thu muc VadCLIP/src_rescale_ewc/ tu repo cuc bo.')
print('\nDu file, va code la ban moi.')

## 4. Copy Feature Sang Runtime Local — BẮT BUỘC

Feature trên Drive nằm ở dạng **file nén**, không phải thư mục. Cell này giải nén nó ra
`/content`. Không chạy cell này thì cả huấn luyện lẫn chấm điểm đều lỗi "missing feature
files" — đây là lỗi đã xảy ra một lần ở vòng 1.

Chạy lại cell này sau **mỗi lần** runtime khởi động lại.

In [ ]:
import shutil, time

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)
start = time.time()
if archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copying archive:', archive)
        shutil.copy2(archive, local_archive)
    print('Extracting:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive phai chua thu muc top-level UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-ah', '--info=progress2',
                        f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'\nDone in {time.time() - start:.1f}s. FEATURE_ROOT =', FEATURE_ROOT)

### 4.1. Sinh Lại Ground Truth (chỉ khi mục 3 báo thiếu)

Ba file `gt_*.npy` không nằm trong repo — chúng được sinh ra từ `Temporal_Anomaly_Annotation.txt`
cộng với **độ dài thật của từng file đặc trưng**, nên bắt buộc phải có feature trước. Đó là lý
do cell này đứng sau mục 4.

`gt_ucf.npy` là nhãn theo khung hình nối tiếp nhau theo đúng thứ tự file list. `gt_segment_ucf.npy`
và `gt_label_ucf.npy` là các đoạn thời gian bất thường, dùng để tính mAP.

Cell tự bỏ qua nếu cả ba file đã có.

In [ ]:
if GT_MISSING:
    run_command(PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', str(LIST_DIR),
    ], log_name='make_gt.log')
else:
    print('Da co du ba file ground truth, bo qua.')

## 5. Unit Test

18 bài, vài giây, không cần feature. Hai bài quan trọng nhất khẳng định khi μ = 1 thì hai
hàm mất mát ở đây bằng đúng từng bit với bản gốc trong `ucf_train.py`. Có bài nào FAIL thì
dừng, đừng huấn luyện.

In [ ]:
run_command(PY + ['tests/test_losses.py'], log_name='test_losses.log')

## 6. Seed 234 — Quét Hệ Số Khuếch Đại

`ctrl` là lần chạy đối chứng: huấn luyện tiếp 3 epoch mà **không can thiệp gì**. Mọi chênh
lệch phải tính so với nó, không phải so với mô hình gốc, nếu không bạn gộp cả hiệu ứng của
việc huấn luyện thêm 3 epoch vào thành công của phương pháp.

Dải μ là **3, 5, 8, 12**. Vòng 1 dùng 2, 3, 5 và phát hiện ra μ=5 nằm dưới ngưỡng phân giải:
nó cho +1,11 ở seed 234 nhưng chỉ +0,09 ở seed 1234. Phải tới μ=8 hiệu ứng mới lặp lại được.
μ=12 là điểm mới, để tìm chỗ đường cong gãy.

`train_stage2` tự bỏ qua lần chạy nào đã có file kết quả, nên nếu runtime đứt giữa chừng thì
chạy lại cell là nó tiếp tục từ chỗ dở.

In [ ]:
train_stage2('ctrl', rescale_mode='off', mu=1.0)

for mu in [3.0, 5.0, 8.0, 12.0]:
    print('=' * 100)
    train_stage2(f'class_mu{mu:g}', rescale_mode='class', mu=mu)

### 6.1. Một Lần Chạy Chế Độ Video Để Đối Chiếu

Vòng 1 đã kết luận chắc chắn chế độ lớp thắng chế độ video: AP trên nhóm đích của chế độ
video **đi xuống** khi μ tăng (+0,13 → +0,03 → −0,49), đúng như bài báo gốc cảnh báo là
khuếch đại cả mẫu thì khuếch đại luôn nhiễu trong mẫu đó.

Nên vòng này chỉ cần **một** lần chạy để tái hiện kết luận, đặt ở μ=8 là chỗ chế độ lớp
hoạt động tốt nhất.

In [ ]:
train_stage2('video_mu8', rescale_mode='video', mu=8.0)

## 7. Seed 1234 — Phép Lặp

Đây là phần bắt buộc, không phải tùy chọn. Vòng 1 cho thấy nhiễu giữa hai seed là ~1,2 điểm
trên chỉ số nhóm đích — gấp ba lần ước lượng ban đầu của tôi, vốn sai vì ba lần chạy dùng
chung một seed và do đó tương quan với nhau.

**Mỗi seed phải có `ctrl` riêng.** So một lần chạy seed 1234 với `ctrl` của seed 234 là vô
nghĩa.

Chỉ lặp hai cấu hình mạnh nhất, không lặp cả dải — mục đích là đo nhiễu, không phải vẽ lại
đường cong.

In [ ]:
train_stage2('ctrl_s1234', rescale_mode='off', mu=1.0, seed=1234)
train_stage2('class_mu8_s1234', rescale_mode='class', mu=8.0, seed=1234)
train_stage2('class_mu12_s1234', rescale_mode='class', mu=12.0, seed=1234)

## 8. Chấm Điểm

Bảng có hai nửa: bên trái là những chỉ số **không được xấu đi**, bên phải là những chỉ số
**cần tăng**. Bố cục theo đúng bảng kết quả của bài báo gốc.

Bảng delta mà script in ra lấy mốc là `source`. Đó **không** phải cái bạn cần — cell 8.1 bên
dưới tính lại theo mốc đúng.

In [ ]:
model_specs = [
    f'source={STAGE1_MODEL}',
    'ctrl=model/v2_ctrl.pth',
    'class_mu3=model/v2_class_mu3.pth',
    'class_mu5=model/v2_class_mu5.pth',
    'class_mu8=model/v2_class_mu8.pth',
    'class_mu12=model/v2_class_mu12.pth',
    'video_mu8=model/v2_video_mu8.pth',
    'ctrl_s1234=model/v2_ctrl_s1234.pth',
    'class_mu8_s1234=model/v2_class_mu8_s1234.pth',
    'class_mu12_s1234=model/v2_class_mu12_s1234.pth',
    # Tuy chon, chi ton tai neu ban chay muc 10:
    'class_mu8_ewc=model/v2_class_mu8_ewc.pth',
]
model_specs = [s for s in model_specs if Path(s.split('=', 1)[1]).exists()]

run_command(PY + [
    'ucf_eval_perclass.py',
    '--feature-root', FEATURE_ROOT,
    '--test-list', TEST_LIST,
    *GT_ARGS,
    '--target-classes', *TARGET_CLASSES,
    '--eval-model-paths', *model_specs,
    '--eval-output', str(PERCLASS_CSV),
], log_name='eval_perclass.log')

### 8.1. Phân Tích Ghép Cặp Theo Seed

Cell này làm ba việc mà bảng thô ở trên không làm.

**Tính delta so với `ctrl` CÙNG SEED.** Đây là phép so sánh đúng. So với `source` thì gộp cả
hiệu ứng huấn luyện thêm; so với `ctrl` của seed khác thì gộp cả nhiễu seed.

**Ghép cặp cùng cấu hình giữa hai seed.** Mức tăng tuyệt đối dao động mạnh giữa các seed,
nhưng ở vòng 1 mức **tăng thêm theo liều** lại lặp lại gần như trùng khít (+0,85 và +0,82 khi
đi từ μ=5 lên μ=8). Đó mới là bằng chứng có sức nặng.

**Lọc bỏ Abuse và Assault khỏi trung bình nhóm không đích.** Hai lớp này chỉ có 2 và 3 video
kiểm tra, dao động tới 3,9 điểm giữa các lần chạy đáng lẽ cho cùng kết quả. Ở vòng 1 chúng
đã làm sai lệch một kết luận trung gian.

In [ ]:
import pandas as pd

TINY_CLASSES = {'Abuse', 'Assault'}   # 2 va 3 video test -> khong dung duoc

summary = pd.read_csv(str(PERCLASS_CSV).replace('.csv', '_summary.csv'))
perclass = pd.read_csv(PERCLASS_CSV)


def ctrl_of(run):
    return 'ctrl_s1234' if run.endswith('_s1234') else 'ctrl'


# --- Chi so tong hop, delta so voi ctrl cung seed ---
s = summary.set_index('run')
rows = []
for run in s.index:
    if run in ('source', 'ctrl', 'ctrl_s1234'):
        continue
    base = s.loc[ctrl_of(run)]
    rows.append({
        'run': run, 'seed': 1234 if run.endswith('_s1234') else 234,
        'd_tgtAUC_C': s.loc[run, 'target_auc_c'] - base['target_auc_c'],
        'd_tgtAP_C': s.loc[run, 'target_ap_c'] - base['target_ap_c'],
        'd_tgtAUC_A': s.loc[run, 'target_auc_a'] - base['target_auc_a'],
        'd_restAUC': s.loc[run, 'rest_auc_c'] - base['rest_auc_c'],
        'd_AUC_C': s.loc[run, 'classifier_auc'] - base['classifier_auc'],
        'd_FPR': s.loc[run, 'normal_fpr@0.5'] - base['normal_fpr@0.5'],
    })
delta = pd.DataFrame(rows).set_index('run')
print('=== Delta so voi ctrl CUNG SEED ===')
print(delta.round(2).to_string())

# --- Nhom khong dich sau khi loc hai lop qua nho ---
c = perclass[(perclass.branch == 'classifier') & (perclass.label != 'Normal')]
p = c.pivot_table(index='label', columns='run', values='auc')
rest_ok = [l for l in p.index
           if l not in set(TARGET_CLASSES) and l not in TINY_CLASSES]
print('\n=== Nhom khong dich, BO Abuse va Assault ===')
for run in delta.index:
    d = (p[run] - p[ctrl_of(run)]).loc[rest_ok].mean()
    print(f'  {run:<20} {d:+.2f}')

# --- Duong dap ung theo lieu, tung seed ---
print('\n=== Duong dap ung theo lieu (AUC nhom dich) ===')
for seed, suffix in ((234, ''), (1234, '_s1234')):
    pts = [(mu, delta.loc[f'class_mu{mu:g}{suffix}', 'd_tgtAUC_C'])
           for mu in (3, 5, 8, 12) if f'class_mu{mu:g}{suffix}' in delta.index]
    if pts:
        print(f'  seed {seed}: ' + '  '.join(f'mu={m:g}: {v:+.2f}' for m, v in pts))

# --- Muc tang them theo lieu, ghep cap giua hai seed ---
print('\n=== Muc tang them giua cac muc mu (chi so lap lai duoc) ===')
for a, b in ((5, 8), (8, 12)):
    line = []
    for seed, suffix in ((234, ''), (1234, '_s1234')):
        ta, tb = f'class_mu{a}{suffix}', f'class_mu{b}{suffix}'
        if ta in delta.index and tb in delta.index:
            line.append(f'seed {seed}: {delta.loc[tb, "d_tgtAUC_C"] - delta.loc[ta, "d_tgtAUC_C"]:+.2f}')
    if line:
        print(f'  mu {a} -> {b}:  ' + '   '.join(line))

# --- Explosion va Shooting: lop phan ung manh nhat va lop bi hai ---
print('\n=== AUC tung lop dang chu y (delta so voi ctrl cung seed) ===')
for lab in ['Explosion', 'Arson', 'Shoplifting', 'RoadAccidents', 'Shooting']:
    if lab in p.index:
        vals = {r: round(float(p.loc[lab, r] - p.loc[lab, ctrl_of(r)]), 2) for r in delta.index}
        print(f'  {lab:<14}', vals)

### 8.2. Cách Đọc Kết Quả

**Chỉ số chính là `d_tgtAUC_C`**, không phải `d_AUC_C`. Ở vòng 1, mười sáu lần chạy có AUC
tổng thể nằm gọn trong 0,37 điểm — chỉ số đó không phân biệt được cấu hình nào với cấu hình
nào và không nên dùng làm kết luận.

**Bằng chứng mạnh nhất là mức tăng thêm theo liều**, không phải giá trị tuyệt đối. Vòng 1:
giá trị tuyệt đối ở μ=8 là +1,96 và +0,91 tùy seed — chênh nhau một điểm. Nhưng mức tăng
thêm từ μ=5 lên μ=8 là +0,85 và +0,82 — trùng khít. Nếu vòng này cho ra hai con số gần nhau
ở dòng "muc tang them" thì bạn có kết quả thật.

**Đường cong gãy ở đâu.** Nếu μ=12 thấp hơn μ=8 thì bạn đã tìm được đỉnh và có một đường
cong hoàn chỉnh — đó là hình thuyết phục nhất mà thí nghiệm này có thể cho. Nếu μ=12 vẫn cao
hơn thì đường cong chưa bão hòa, và phải nhìn cột `d_FPR`: ở vòng 1 báo động giả đã tăng
0,5–0,9 điểm ngay từ μ=8.

**Shooting là cái giá.** Vòng 1 cho thấy nó bị hại ở 7/7 lần chạy có μ≥5, và mức hại tăng
theo μ. Đây là kết quả lặp lại tốt nhất của cả thí nghiệm, trớ trêu thay lại là mặt tiêu cực.
Phải báo cáo, đừng giấu.

## 9. (Tùy chọn) Tái Hiện Kết Quả Âm Tính Về Fisher

Vòng 1 kết luận thành phần Fisher **không đóng góp gì**: ba cơ chế bảo vệ cho 62,41 / 62,63 /
62,79 trên chỉ số nhóm đích, tức không phân biệt được. Nguyên nhân đã xác định — độ trôi
trọng số chỉ khoảng 1e-5, mô hình gần như đứng yên, nên một cơ chế chống trôi không có việc
gì làm. Bài báo gốc cần nó vì họ tinh chỉnh trên dữ liệu ngoài miền và bị quên nghiêm trọng.

Đây là một **kết quả âm tính có giá trị** và có trong báo cáo, nên nếu bạn muốn nó tái hiện
được thì chạy hai cell dưới. Nếu chỉ cần kết quả chính thì bỏ qua — nó tốn thêm một lần ước
lượng Fisher (1000 lượt forward+backward) cộng một lần huấn luyện.

In [ ]:
run_command(PY + [
    'ucf_fisher.py',
    '--pretrained-model-path', STAGE1_MODEL,
    '--feature-root', FEATURE_ROOT,
    '--train-list', TRAIN_LIST,
    '--fisher-path', FISHER_PATH,
    '--fisher-max-samples', 1000,
    '--fisher-log-every', 100,
], log_name='fisher.log')

Đọc bảng thống kê Fisher: **kỳ vọng phân bố rất lệch**. Vòng 1 cho trung vị khoảng 0,07 trong
khi giá trị lớn nhất vượt 18.000 sau khi chuẩn hóa. Chính khoảng cách đó là toàn bộ khác biệt
giữa phạt theo Fisher và phạt đều tay — nếu nó phẳng thì hai cơ chế trùng nhau về mặt toán
học và không cần chạy cell dưới.

In [ ]:
train_stage2('class_mu8_ewc', rescale_mode='class', mu=8.0,
             regularizer='ewc', lambda_auto=0.15)